In [1]:
#number interpretable nuerons

import pandas as pd
import re
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

import os
def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts



def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)

    unit_concepts = defaultdict(set)
    raw_concepts= []
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        
        unit_concepts[unit].update(concepts)
        raw_concepts.extend(concepts)
        
    return unit_concepts


def get_all_cps_for_pi(folder):
    root_path = Path(folder)

    # Find all matching CSV files
    csv_pattern = 'Cluster*IOUS1024N.csv'
    csv_files = list(root_path.rglob(csv_pattern))
    
    concept_dict=defaultdict(set)
    for csv_file in csv_files:
        concepts = []
        csv_file = os.path.join(folder, csv_file)
        df = pd.read_csv(csv_file)
        for unit, formula in zip(df.unit, df.best_name):
            concept_dict[unit].update(get_indiv_concepts(formula))
    return concept_dict

def get_cluster_cps_for_pi(folder, cluster):
    csv_file = f'Cluster{cluster}IOUS1024N.csv'
    
    concept_dict=defaultdict(set)
    concepts = []
    csv_file = os.path.join(folder, csv_file)
    df = pd.read_csv(csv_file)
    for unit, formula in zip(df.unit, df.best_name):
        concept_dict[unit].update(get_indiv_concepts(formula))
    return concept_dict

In [6]:
models=['LLAMA', 'BOWMAN', 'BERT']
methods=['lottery_ticket', 'CoFi']

for model in models:
    for method in methods:
        for i in range(1,4):
            densePath = f'/workspace/CCE_NLI/{model}/{method}/Run0.25_5/Expls/0.0%Pruned/Cluster{i}IOUS1024N.csv'
            denseeplxs = pd.read_csv(densePath)
            print(model, method, i, len(denseeplxs['unit'])/1024)

LLAMA lottery_ticket 1 0.9970703125
LLAMA lottery_ticket 2 0.990234375
LLAMA lottery_ticket 3 0.6953125
LLAMA CoFi 1 1.0
LLAMA CoFi 2 1.0
LLAMA CoFi 3 0.9814453125
BOWMAN lottery_ticket 1 0.0166015625
BOWMAN lottery_ticket 2 0.01171875
BOWMAN lottery_ticket 3 0.001953125
BOWMAN CoFi 1 0.0263671875
BOWMAN CoFi 2 0.0166015625
BOWMAN CoFi 3 0.005859375
BERT lottery_ticket 1 1.0
BERT lottery_ticket 2 1.0
BERT lottery_ticket 3 0.912109375
BERT CoFi 1 1.0
BERT CoFi 2 1.0
BERT CoFi 3 0.95703125
